# Ukrainian FastText: initialization and subword OOV

Three fresh 300d BiLSTMs: random, FastText with shared UNK, and FastText with consistent training/inference OOV vectors. One frozen n-grams reference. Read `docs/FASTTEXT_PROTOCOL.md` before training. Upload the prepared `fasttext_current_source/` folder next to the existing `data/` and `results/`. Keep `domain_shift_source/` and all previous runs unchanged.

The official archive is 4.50 GB; its matrices alone need 7.20 GB RAM. First run cells 1–4 in a CPU runtime to cache vectors. If available RAM is below the explicit check, choose a High-RAM CPU runtime. Then switch to a GPU and Run All; prepared vectors are reused. CUDA training never falls back to CPU. No Kaggle submission or audit evaluation is performed. New outputs: `results/fasttext_reproduction/`. A/B and the new subword stage each have a resumable 45-minute session cap; this is a limit, not a measured runtime. Completed runs are reused.

This local notebook reproduces the corrected Git source in a new directory. Historical training source and results remain unchanged. For the trained checkpoint submission, use `fasttext_submission.py` separately.

## 1. Drive and verified ordinary source files

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, shutil, sys, subprocess

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/KSE_AI301_A3')
WORK = Path('/content/a3_fasttext_v1')
WORK.mkdir(parents=True, exist_ok=True)
SOURCE = ROOT / 'fasttext_current_source'
MANIFEST_SHA = 'd68d5665628e3d69e12e2de213eb3cd04d29a18c3fba9e6026a68b09ba402572'
raw = (SOURCE / 'source_manifest.json').read_bytes()
assert hashlib.sha256(raw).hexdigest() == MANIFEST_SHA, 'Source manifest changed'
manifest = json.loads(raw)
LOCAL = WORK / 'source'
for name, expected in manifest['files'].items():
    assert not Path(name).is_absolute() and '..' not in Path(name).parts
    assert hashlib.sha256((SOURCE / name).read_bytes()).hexdigest() == expected, name
    target = LOCAL / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SOURCE / name, target)
shutil.copy2(SOURCE / 'source_manifest.json', LOCAL / 'source_manifest.json')
os.environ['MPLCONFIGDIR'] = str(WORK / 'matplotlib')
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
sys.path.insert(0, str(LOCAL))


## 2. Dependencies, W&B and local fixture preflight

In [ ]:
import torch, importlib.metadata
original_torch = importlib.metadata.version('torch')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LOCAL / 'requirements-fasttext.txt')])
assert importlib.metadata.version('torch') == original_torch
from google.colab import userdata
try:
    if not os.environ.get('WANDB_API_KEY'):
        key = userdata.get('WANDB_API_KEY')
        if key: os.environ['WANDB_API_KEY'] = key
except Exception as error:
    print('W&B secret unavailable; offline tracking:', type(error).__name__)
print('CUDA:', torch.cuda.is_available(), 'W&B project: dtkach-kse/ai301-a3')
subprocess.check_call([sys.executable, '-m', 'unittest', 'test_fasttext', 'test_fasttext_subword', '-v'], cwd=LOCAL)
from experiments import fasttext_experiment as experiment
from experiments.fasttext_embeddings import prepare_vectors
from experiments import fasttext_subword as subword
SOURCE_ID = experiment.verify_source(LOCAL, LOCAL / 'source_manifest.json')


## 3. Frozen data and read-only reference

In [ ]:
DATA = WORK / 'data'
DATA.mkdir(exist_ok=True)
spec = json.loads((LOCAL / 'part2_split.json').read_text())
from part2_data import file_hash
for name, expected in spec['files'].items():
    target = DATA / name
    if not target.exists() or file_hash(target) != expected:
        shutil.copy2(ROOT / 'data' / name, target)
REFERENCE = ROOT / 'results/word_representations_v1/ngrams/main/best.pt'
OUTPUT = ROOT / 'results/fasttext_reproduction'
docs, split, words, checks = experiment.preflight(DATA, LOCAL / 'part2_split.json', REFERENCE, OUTPUT, subword.primary_source(OUTPUT, SOURCE_ID))
print(checks)


## 4. Verify official FastText and cache small embedding tables

In [ ]:
PREPARED, vector_info = prepare_vectors(words, docs['fit'], docs['checkpoint'] + docs['calibration'],
    ROOT / 'cache/fasttext', WORK / 'fasttext')
print(vector_info['verification'])
print(vector_info['coverage'])
SUBWORDS, subword_info = subword.prepare_subwords(words, docs['fit'], docs['checkpoint'] + docs['calibration'],
    vector_info, ROOT / 'cache/fasttext', WORK / 'fasttext')
print('Fixed OOV cache:', subword_info)
print('OOV buffer MiB:', subword_info['buffer_bytes'] / 1024**2)
# The large binary model is released before either NER model is initialized.


## 5. Train A/B and the separate subword model on Colab GPU

In [ ]:
results = subword.run_primary(docs, split, words, PREPARED, vector_info,
    OUTPUT, WORK / 'training', SOURCE_ID, minutes=45)
if results is None:
    raise RuntimeError('Training incomplete: review session_status.json and resume with Run All')
subword_result = subword.run_subword(docs, split, words, PREPARED, vector_info, SUBWORDS, subword_info,
    OUTPUT, WORK / 'training', SOURCE_ID, minutes=45)
if subword_result is None:
    raise RuntimeError('Subword training incomplete; resume with Run All')
print({name: dict(epochs=r['epochs'], best_epoch=r['best_epoch'], parameters=r['parameters'])
       for name,r in dict(results, fasttext_subword=subword_result).items()})


## 6. Locked checkpoint/calibration comparison

In [ ]:
report = experiment.evaluate(docs, split, words, REFERENCE, PREPARED, OUTPUT, device='cuda')
print('Source-domain initialization benefit:', report['source_domain_initialization_benefit'])
print('Matched precision protocol:', report['protocol_precision_matches'])
print('Comparison:', OUTPUT / 'comparison.csv')
print('Confidence intervals:', OUTPUT / 'bootstrap.json')
subword_report = subword.evaluate_subword(docs, split, words, PREPARED, SUBWORDS, subword_info, OUTPUT, REFERENCE, device='cuda')
import pandas as pd
display(pd.read_csv(OUTPUT / 'subword/comparison.csv'))
print('New comparison:', OUTPUT / 'subword/comparison.json')
print('The old inference-only OOV diagnostic is not the trained subword model.')
# Baseline is never replaced. No audit scoring or Kaggle submission.
